# Perfil de Qualificação da Audiência

Esse notebook constroe um score de qualificação da audiência para cada perfil, baseado em duas dimensões comportamentais observáveis: nível de engajamento típico e estabilidade do engajamento ao longo do tempo.

## Definição operacional

A qualificação da audiência é tratada como um representante relativo ao segmento do perfil.
O score não mede demografia, autenticidade direta de seguidores ou variáveis latentes de qualidade.
Trata-se de um indicador de posicionamento relativo ao segmento (`inf_category × bucket_followers`),
o que torna a comparação metodologicamente justa entre perfis de porte e nicho semelhantes.


1. **Nível de engajamento típico** (mediana do ER clássico do perfil);
2. **Estabilidade do engajamento** (coeficiente de variação invertido).

## Dimensões do score

| Dimensão | Indicador | Direção | Peso |
|---|---|---|---|
| Engajamento típico | Mediana do ER clássico do perfil | ↑ maior = melhor | 0,60 |
| Estabilidade | CV do ER | ↓ menor CV = melhor | 0,40 |

Na seção 07 há uma análise de sensibilidade dos pesos(0,60/0,40).

## Normalização por segmento

Cada dimensão é normalizada como rank percentil dentro do segmento, de modo que um perfil de fashion nano so é comparado com outros perfis de fashion nano, cujo ER estrutural são similares.

## 1. Configuração

In [39]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr

pl.Config.set_tbl_rows(10)
pl.Config.set_fmt_str_lengths(60)
sns.set_theme(style="whitegrid", context="talk")

In [2]:
# Caminhos
DIRETORIO = Path("bases-processadas/")

PARQUET_BASE = f"{DIRETORIO}/posts_base_2017_2019.parquet"
PARQUET_CV = f"{DIRETORIO}/cv_base_2017_2019.parquet"   
PARQUET_CV_ROB = f"{DIRETORIO}/cv_core_2017_2019.parquet"   

OUT_PARQUET = f"{DIRETORIO}/perfil_qualificacao.parquet"
OUT_CSV = f"{DIRETORIO}/perfil_qualificacao.csv"

ORDEM_BUCKET = ["nano", "micro", "mid-tier", "macro", "mega"]
ORDEM_TIER = ["D", "C", "B", "A"]

# Pesos do score
W_ER = 0.60
W_CV = 0.40

## 2. Carga e Verificação das bases

| Base | Descrição |
|---|---|
| `posts_base_2017_2019` | Base analítica principal — nível de post, 2017–2019, followers > 0 |
| `cv_base_2017_2019` | CV do ER clássico por perfil — calculado sobre todos os posts válidos (**base principal**) |
| `cv_core_2017_2019` | CV calculado apenas sobre posts com `comments_count > 0` — usado na verificação de robustez |


In [ ]:
base   = pl.read_parquet(PARQUET_BASE)
cvbase = pl.read_parquet(PARQUET_CV)
# cvcore = pl.read_parquet(PARQUET_CV_ROB)

print(f"Base principal:{base.shape}")
print(f"CV base:{cvbase.shape}")
# print(f"CV core: {cvcore.shape}")

Base principal:(8897233, 32)
CV base:(33127, 6)
CV core: (32646, 6)


## 3. Agregação em nível de perfil

Cada perfil é resumido pela mediana do ER classico. A média é mantida como métrica de auxiliar.

In [4]:
perfis = (
    base.group_by("username")
    .agg([
        pl.len().alias("n_posts_total"),
        pl.col("followers").median().alias("followers"),
        pl.col("inf_category").first().alias("inf_category"),
        pl.col("bucket_followers").first().alias("bucket_followers"),
        pl.col("perfil_completo_lookup").first().alias("perfil_completo_lookup"),
        pl.col("er_classico").median().alias("erclassico_mediana"),
        pl.col("er_classico").mean().alias("erclassico_media"),
        pl.col("er_weighted").median().alias("erweighted_mediana"),
        pl.col("is_sponsored").mean().alias("is_sponsored_pct"),
        (pl.col("post_type") == "image").mean().alias("pct_image"),
        (pl.col("post_type") == "carousel").mean().alias("pct_carousel"),
    ])
)


In [5]:
perfis = perfis.join(cvbase.select(["username", "n_posts_validos", "er_media", "er_mediana", "cv"]),on="username", how="left")

In [6]:
perfis

username,n_posts_total,followers,inf_category,bucket_followers,perfil_completo_lookup,erclassico_mediana,erclassico_media,erweighted_mediana,is_sponsored_pct,pct_image,pct_carousel,n_posts_validos,er_media,er_mediana,cv
str,u32,f64,str,str,bool,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64
"""camilomontoya""",300,91926.0,"""family""","""micro""",true,0.683702,0.899662,0.686422,0.0,0.99,0.01,300,0.899662,0.683702,0.772349
"""marthashappilyeverafter""",300,39262.0,"""interior""","""micro""",true,3.914727,4.137427,3.928735,0.23,0.963333,0.036667,300,4.137427,3.914727,0.422518
"""true.facts.official""",25,75244.0,"""other""","""micro""",true,5.362554,9.067726,5.362554,0.0,1.0,0.0,25,9.067726,5.362554,0.865458
"""808projectd""",300,1280.0,"""other""","""nano""",true,1.25,1.393229,1.2890625,0.003333,0.893333,0.106667,300,1.393229,1.25,0.479176
"""anna.rana""",200,14876.0,"""travel""","""micro""",true,1.317558,3.07102,1.317558,0.04,0.975,0.025,200,3.07102,1.317558,1.269942
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ayla_woodruff""",251,1.492855e6,"""fashion""","""mega""",true,6.449052,6.467984,6.449052,0.075697,0.888446,0.111554,251,6.467984,6.449052,0.509597
"""gabpacifico""",300,5721.0,"""fashion""","""nano""",true,3.688166,3.841636,3.731865,0.03,0.876667,0.123333,300,3.841636,3.688166,0.291066
"""carlottiica""",181,30517.0,"""fashion""","""micro""",true,3.224432,3.684786,3.227709,0.033149,0.834254,0.165746,181,3.684786,3.224432,0.602459


## 4. Critérios de Elegibilidade

Um perfil precisa satisfazer todos os critérios abaixo para entrar no cálculo do score.
Perfis inelegíveis são preservados na base final com `elegivel = False` e `score_audiencia = null`,
garantindo rastreabilidade.

| Critério | Limiar | Justificativa |
|---|---|---|
| Posts totais | ≥ 30 | Garantia de representatividade temporal do perfil no período |
| Posts válidos no CV | ≥ 10 | Mínimo para estimativa estável do coeficiente de variação |
| Lookup completo | `perfil_completo_lookup == True` | Perfis sem dados do lookup têm followers = 0 |
| Categoria e bucket presentes | não nulo | Necessário para normalização por segmento |

In [12]:
total = perfis.shape[0]
sem_cv           = perfis.filter(pl.col("cv").is_null()).shape[0]
menos_10_validos = perfis.filter(pl.col("n_posts_validos") < 10).shape[0]
menos_30_posts = perfis.filter(pl.col("n_posts_total") < 30).shape[0]
lookup_inc       = perfis.filter(pl.col("perfil_completo_lookup") == False).shape[0]
cat_nula         = perfis.filter(pl.col("inf_category").is_null()).shape[0]
bucket_nulo      = perfis.filter(pl.col("bucket_followers").is_null()).shape[0]

In [13]:
inelegivel = perfis.filter(
    (pl.col("n_posts_total") < 30) |
    (pl.col("n_posts_validos") < 10) |
    (pl.col("perfil_completo_lookup") == False) |
    pl.col("inf_category").is_null() |
    pl.col("bucket_followers").is_null() |
    pl.col("cv").is_null()
).shape[0]

In [15]:
print("Diagnostico dos perfis: ")

print(f"Total de perfis: {total}")
print(f"Menos de 30 posts: {menos_30_posts}")
print(f"Menos de 10 posts válidos para CV: {menos_10_validos}")
print(f"Lookup Incompleto: {lookup_inc}")
print(f"Categoria incompleta: {cat_nula}")
print(f"Bucket Ausente: {bucket_nulo}")
print(f"CV Ausente: {sem_cv}")
print(f"Total de perfis não elegiveis: {inelegivel}")

Diagnostico dos perfis: 
Total de perfis: 33149
Menos de 30 posts: 140
Menos de 10 posts válidos para CV: 24
Lookup Incompleto: 0
Categoria incompleta: 0
Bucket Ausente: 0
CV Ausente: 22
Total de perfis não elegiveis: 140


- Aplicar uma flag de elegibilidade

In [16]:
perfis = perfis.with_columns(
    pl.when(
        (pl.col("n_posts_total") >= 30) &
        (pl.col("n_posts_validos") >= 10) &
        (pl.col("perfil_completo_lookup") == True) &
        pl.col("inf_category").is_not_null() &
        pl.col("bucket_followers").is_not_null() &
        pl.col("cv").is_not_null()
    ).then(True).otherwise(False).alias("elegivel")
)

elegiveis = perfis.filter(pl.col("elegivel") == True)
print(f"Perfis elegíveis:   {elegiveis.shape[0]}")
print(f"Perfis inelegíveis: {perfis.filter(pl.col('elegivel') == False).shape[0]}")

Perfis elegíveis:   33009
Perfis inelegíveis: 140


## 5. Construção do Score de qualificação

Para cada perfil elegível, calcula-se o rank percentil de er_mediana e de cv dentro do seu segmento (categoria x bucket de followers)

O rank percentil \( p_i \) de um perfil \( i \) dentro do segmento \( s \) é definido como:

$
p_i = \frac{\text{rank}(x_i) - 1}{N_s - 1}
$

onde \( N_s \) é o número de perfis elegíveis no segmento e $( \text{rank}(x_i) )$ é a posição
ordinal de \( i \) (1 = menor valor).

O score final é então:

$
\text{score}_i = W_{ER} \cdot p_{ER,i} + W_{CV} \cdot (1 - p_{CV,i})
$

onde $( W_{ER} = 0{,}60 )$ e $( W_{CV} = 0{,}40 )$. O CV é invertido porque menor CV
(maior estabilidade) é desejável.


In [17]:
elegiveis_pd = elegiveis.to_pandas()

In [ ]:
def rank_percentil(series):
    n = len(series)
    if n == 1:
        return pd.Series([0.5], index=series.index)
    return series.rank(method="average").sub(1).div(n - 1)

In [20]:
elegiveis_pd["rank_er"] = (
    elegiveis_pd.groupby(["inf_category", "bucket_followers"])["erclassico_mediana"]
    .transform(rank_percentil)
)
elegiveis_pd["rank_cv"] = (
    elegiveis_pd.groupby(["inf_category", "bucket_followers"])["cv"]
    .transform(rank_percentil)
)

- Score final

In [21]:
elegiveis_pd["score_audiencia"] = (
    W_ER * elegiveis_pd["rank_er"] +
    W_CV * (1 - elegiveis_pd["rank_cv"])
)

In [27]:
elegiveis_pd.sort_values("score_audiencia",ascending=False)

,username,n_posts_total,followers,inf_category,bucket_followers,perfil_completo_lookup,erclassico_mediana,erclassico_media,erweighted_mediana,is_sponsored_pct,pct_image,pct_carousel,n_posts_validos,er_media,er_mediana,cv,elegivel,rank_er,rank_cv,score_audiencia
3962,foodiemuggles,298,4252.0,food,nano,True,36.876764,36.600857,36.935560,0.033557,0.409396,0.590604,298,36.600857,36.876764,0.135477,True,1.000000,0.000000,1.000000
21944,itsdougthepug,299,3642341.0,pet,mega,True,3.142704,3.310001,3.142704,0.020067,0.923077,0.076923,299,3.310001,3.142704,0.304496,True,1.000000,0.000000,1.000000
10767,nico.sunshine.50,299,1377.0,travel,nano,True,26.579521,26.738608,26.652142,0.000000,1.000000,0.000000,299,26.738608,26.579521,0.129958,True,0.998250,0.001167,0.998483
24094,dailysurfgr,254,3408.0,other,nano,True,23.474178,23.038081,23.474178,0.007874,0.818898,0.181102,254,23.038081,23.474178,0.139913,True,0.997543,0.000351,0.998385
17173,askchefdennis,300,30394.0,food,micro,True,14.673949,14.916738,14.739751,0.053333,1.000000,0.000000,300,14.916738,14.673949,0.115664,True,0.998842,0.002895,0.998147
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12361,thestylejournal_,300,166317.0,fashion,mid-tier,True,0.065838,0.247706,0.065838,0.020000,0.810000,0.190000,300,0.247706,0.065838,2.304746,True,0.000557,0.998886,0.000780
26122,peachyqueenblog,294,1349883.0,beauty,mega,True,0.099972,0.176874,0.099972,0.000000,0.755102,0.244898,294,0.176874,0.099972,1.537523,True,0.000000,1.000000,0.000000
17440,dogsofinstagram,300,4284911.0,pet,mega,True,0.960288,1.073313,0.960288,0.036667,0.873333,0.126667,300,1.073313,0.960288,0.509256,True,0.000000,1.000000,0.000000
32738,paperazzimagazine,300,588477.0,fashion,macro,True,0.021666,0.063588,0.021666,0.000000,0.726667,0.273333,300,0.063588,0.021666,2.259413,True,0.000000,1.000000,0.000000


- Cobertura dos segmentos

In [30]:
cobertura = (
    elegiveis_pd.groupby(["inf_category", "bucket_followers"])
    .size()
    .reset_index(name="n_perfis")
    .pivot(index="inf_category", columns="bucket_followers", values="n_perfis")
)
cobertura = cobertura[[b for b in ORDEM_BUCKET if b in cobertura.columns]].fillna(0).astype(int)
cobertura

bucket_followers,nano,micro,mid-tier,macro,mega
inf_category,,,,,
beauty,524,693,210,34,37
family,1806,1596,368,76,101
fashion,3050,6103,1796,324,353
fitness,285,562,188,24,37
food,1460,1728,252,25,6
interior,363,676,115,13,7
other,2850,1821,575,118,164
pet,156,324,80,6,4
travel,1715,1908,402,47,27


## 6. Classiciação em Tiers

Os perfis elegiveis são então classificados em 4 tiers(A,B,C e D) com base em quartis do score, calculados globalmente após normalização por segmento.

| Tier | Quartil | Interpretação |
|---|---|---|
| A | Q4 (75–100%) | Alta qualificação - engajamento elevado e estável no segmento |
| B | Q3 (50–75%) | Qualificação moderada-alta |
| C | Q2 (25–50%) | Qualificação moderada-baixa |
| D | Q1 (0–25%) | Baixa qualificação - engajamento baixo ou instável |


- classificação em tiers por quartil do score

In [32]:
elegiveis_pd["tier_audiencia"] = pd.qcut(
    elegiveis_pd["score_audiencia"],
    q=4,
    labels=["D", "C", "B", "A"]
)

In [33]:
tier_dist = (
    elegiveis_pd.groupby("tier_audiencia", observed=True)
    .size()
    .reset_index(name="n_perfis")
)
tier_dist["pct"] = (tier_dist["n_perfis"] / tier_dist["n_perfis"].sum() * 100).round(2)
tier_dist

,tier_audiencia,n_perfis,pct
0,D,8253,25.0
1,C,8252,25.0
2,B,8252,25.0
3,A,8252,25.0


- Perfil descritivo dos tiers

In [34]:
tier_desc = (
    elegiveis_pd.groupby("tier_audiencia", observed=True)
    .agg(
        erclassico_mediana_mediana=("erclassico_mediana", "median"),
        cv_mediano=("cv", "median"),
        score_mediano=("score_audiencia", "median"),
        n_posts_total_mediano=("n_posts_total", "median"),
        is_sponsored_pct_mediana=("is_sponsored_pct", "median"),
    )
    .round(4)
)
tier_desc

,erclassico_mediana_mediana,cv_mediano,score_mediano,n_posts_total_mediano,is_sponsored_pct_mediana
tier_audiencia,,,,,
D,1.2740,0.7144,0.2000,299.0,0.0254
C,2.4692,0.5083,0.4177,299.0,0.0282
B,3.8621,0.4562,0.5925,299.0,0.0286
A,6.2985,0.3483,0.7924,298.0,0.0267


- Distribuição dos tiers por bucket de seguidores

In [35]:
tier_bucket = (
    elegiveis_pd.groupby(["bucket_followers", "tier_audiencia"], observed=True)
    .size()
    .reset_index(name="n")
    .pivot(index="bucket_followers", columns="tier_audiencia", values="n")
    .fillna(0).astype(int)
)
tier_bucket = tier_bucket.loc[[b for b in ORDEM_BUCKET if b in tier_bucket.index]]

In [36]:
tier_bucket_pct = tier_bucket.div(tier_bucket.sum(axis=1), axis=0).round(3)
tier_bucket_pct

tier_audiencia,D,C,B,A
bucket_followers,,,,
nano,0.254,0.254,0.241,0.251
micro,0.243,0.254,0.258,0.245
mid-tier,0.255,0.235,0.255,0.254
macro,0.271,0.210,0.238,0.280
mega,0.279,0.219,0.215,0.288


- Correlação entre score e dimensões componentes


In [37]:
corr_cols = ["score_audiencia", "erclassico_mediana", "erweighted_mediana", "cv"]
elegiveis_pd[corr_cols].corr().round(4)

,score_audiencia,erclassico_mediana,erweighted_mediana,cv
score_audiencia,1.0000,0.6852,0.685,-0.5113
erclassico_mediana,0.6852,1.0000,1.000,-0.2221
erweighted_mediana,0.6850,1.0000,1.000,-0.2220
cv,-0.5113,-0.2221,-0.222,1.0000


## 7. Análise de Sensibilidade dos pesos

Na ausência de uma ponderação teoricamente estabelecida na literatura, a escolha de
\( W_{ER} = 0{,}60 \) e \( W_{CV} = 0{,}40 \) é uma decisão do pesquisador. Para demonstrar
que os resultados não dependem criticamente dessa escolha, calcula-se o score para
três cenários de pesos e verifica-se:

1. A correlação de Spearman entre os rankings gerados por cada cenário.
2. A proporção de perfis que mantém o mesmo tier independentemente dos pesos.

Se a correlação entre rankings for > 0,95 e a estabilidade de tier for > 85%,
a classificação é considerada robusta à escolha de pesos

In [40]:
# Cenários de pesos para análise de sensibilidade
cenarios = {
    "50_50": (0.50, 0.50),
    "60_40": (0.60, 0.40),  # cenário principal
    "70_30": (0.70, 0.30),
}

scores_cenarios = pd.DataFrame({"username": elegiveis_pd["username"]})

for nome, (w_er, w_cv) in cenarios.items():
    scores_cenarios[f"score_{nome}"] = (
        w_er * elegiveis_pd["rank_er"] +
        w_cv * (1 - elegiveis_pd["rank_cv"])
    )

# Correlações de Spearman entre cenários
print("Correlações de Spearman entre rankings por cenário de pesos:")
pares = [("50_50", "60_40"), ("60_40", "70_30"), ("50_50", "70_30")]
for a, b in pares:
    rho, _ = spearmanr(scores_cenarios[f"score_{a}"], scores_cenarios[f"score_{b}"])
    print(f"  {a} vs {b}: ρ = {rho:.4f}")

Correlações de Spearman entre rankings por cenário de pesos:
  50_50 vs 60_40: ρ = 0.9875
  60_40 vs 70_30: ρ = 0.9896
  50_50 vs 70_30: ρ = 0.9550


In [41]:
# Estabilidade de tier entre cenários
for nome, (w_er, w_cv) in cenarios.items():
    scores_cenarios[f"tier_{nome}"] = pd.qcut(
        scores_cenarios[f"score_{nome}"], q=4, labels=["D", "C", "B", "A"]
    )

# Proporção de perfis que mantém o mesmo tier em todos os cenários
tier_cols = [f"tier_{c}" for c in cenarios]
scores_cenarios["tier_estavel"] = scores_cenarios[tier_cols].apply(
    lambda row: len(set(row)) == 1, axis=1
)

pct_estavel = scores_cenarios["tier_estavel"].mean() * 100
print(f"Perfis com mesmo tier em todos os cenários: {pct_estavel:.2f}%")

# Distribuição de discordâncias
print(f"\nPerfis que mudam de tier:")
print(scores_cenarios[~scores_cenarios["tier_estavel"]][tier_cols].value_counts().head(10))

Perfis com mesmo tier em todos os cenários: 74.94%

Perfis que mudam de tier:
tier_50_50  tier_60_40  tier_70_30
B           C           C             953
C           B           B             949
B           B           C             757
C           C           B             757
B           B           A             713
A           A           B             702
            B           B             622
B           A           A             607
C           C           D             552
D           C           C             548
Name: count, dtype: int64


In [42]:
# Resumo da análise de sensibilidade
print("=== Resumo da análise de sensibilidade ===")
print(f"Cenário principal: W_ER={W_ER}, W_CV={W_CV}")
print(f"Perfis elegíveis analisados: {len(elegiveis_pd)}")
print(f"Estabilidade de tier (todos os cenários): {pct_estavel:.2f}%")
print()
print("Conclusão: se correlações > 0.95 e estabilidade > 85%,")
print("a classificação é robusta à escolha de pesos.")

=== Resumo da análise de sensibilidade ===
Cenário principal: W_ER=0.6, W_CV=0.4
Perfis elegíveis analisados: 33009
Estabilidade de tier (todos os cenários): 74.94%

Conclusão: se correlações > 0.95 e estabilidade > 85%,
a classificação é robusta à escolha de pesos.


## 8. Persistencia da base final

- Montar base final. Perfis inelegiveis tem score/tier nulos

In [43]:
inelegiveis_pd = perfis.filter(pl.col("elegivel") == False).to_pandas()
inelegiveis_pd["rank_er"]        = np.nan
inelegiveis_pd["rank_cv"]        = np.nan
inelegiveis_pd["score_audiencia"] = np.nan
inelegiveis_pd["tier_audiencia"]  = np.nan

In [44]:
colunas_finais = list(inelegiveis_pd.columns)
elegiveis_export = elegiveis_pd[[c for c in colunas_finais if c in elegiveis_pd.columns]].copy()

In [45]:
base_final = pd.concat([elegiveis_export, inelegiveis_pd], ignore_index=True)

- Exportar

In [47]:
base_final_pl = pl.from_pandas(base_final)
base_final_pl.write_parquet(OUT_PARQUET)
base_final_pl.write_csv(OUT_CSV)

print(f"Exportado: {OUT_PARQUET}")
print(f"Exportado: {OUT_CSV}")

Exportado: bases-processadas/perfil_qualificacao.parquet
Exportado: bases-processadas/perfil_qualificacao.csv


## 9. Síntese dos resultados